In [1]:
import kagglehub
import pandas as pd
import numpy as np

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 01. Leitura do Dataset

In [2]:
# Download latest version
path = kagglehub.dataset_download("ranafayezz/titanic-cleaned")

df_train = pd.read_csv(path+"/train_data.csv")
df_test = pd.read_csv(path+"/test_data.csv")

# 02. Construção do Exercício

In [3]:
# Vou gerar uma variável qualitativa ordinal a partir das colunas
# `Pclass_1`, `Pclass_2`, e `Pclass_3` (qualitativas nominais)
df_train['Pclass'] = df_train['Pclass_1'] + 2*df_train['Pclass_2'] + 3*df_train['Pclass_3']
df_test['Pclass'] = df_test['Pclass_1'] + 2*df_test['Pclass_2'] + 3*df_test['Pclass_3']

# Para fins desse exercício, vou escolher três variáveis explicativas:
var_explicativas = [
    "Age",  # Quantitativa discreta
    "Pclass",  # Qualitativa ordinal
    "Sex",  # Qualitativa nominal
]

df_X = df_train[var_explicativas]
df_y = df_train["Survived"]
df_train = df_train[var_explicativas + ["Survived"]]
df_test = df_test[var_explicativas + ["Survived"]]

# 03. Construção do Algoritmo da Árvore

In [ ]:
# Definir função de probabilidade logística
def logistic_function(z):
    z = np.clip(z, a_min=-700, a_max=700) # Evitar overflow em np.exp
    return 1 / (1 + np.exp(-z)) # Esse valor está em termos de probabilidade

# Função de custo (log loss). Vai ser usada para avaliar o modelo durante treino
def log_loss(leaf, x, y):
    p = logistic_function(leaf, x)
    p = np.clip(p, a_min=1e-15, a_max=1 - 1e-15) # Evitar log(0) ou log (1)

    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

In [ ]:
# Primeiro, vou criar uma função para abstrair o algoritmo CART
def find_cart_threshold(df, variable, target):
    # Primeiro passo é definir qual é o melhor threshold dessa variável.
    # Para isso vou calcular a quantidade de sobreviventes e casualidades
    # para cada valor da variável, e em seguida aplicar o mesmo cálculo
    # de log_loss e obter o melhor threshold
    possible_variable_values = df[variable].unique()

    if len(possible_variable_values) == 1:
        return None, 0

    MSEs = {}

    for value in possible_variable_values:
        lt_subset = target[df[variable] <= value]
        gt_subset = target[df[variable] > value]

        if len(lt_subset) == 0 or len(gt_subset) == 0:
            continue

        lt_mse = (lt_subset - lt_subset.mean()) ** 2
        gt_mse = (gt_subset - gt_subset.mean()) ** 2
        mse = (len(lt_subset) * lt_mse.sum() + len(gt_subset) * gt_mse.sum()) / len(df)
        MSEs[value] = mse

    best_threshold = min(MSEs, key=MSEs.get)
    return best_threshold, MSEs[best_threshold]


# Função para abstrair o processo de encontrar o melhor split
def find_best_split(df, target):
    """
    Entre todas as features, escolhe o (feature, threshold) com menor
    log loss gerado. Retorna (None, None, None) se não
    houver split válido.

    `evaluated_features` é um float entre 0 e 1 que indica a fração de features
    que serão avaliadas.
    """

    candidates = {}
    for var in df.columns:

        threshold, mse = find_cart_threshold(df, var, target)
        if threshold is None:
            continue

        candidates[var] = (threshold, mse)

    if not candidates:
        return None, None, None

    best_feature = min(candidates, key=lambda var: candidates[var][1])
    threshold, mse = candidates[best_feature]
    return best_feature, threshold, mse

In [ ]:
# Construir classe que representa um nó na árvore
class Node:
    """Nó de uma árvore de decisão.

    `prediction` guarda a classe majoritária do nó (útil na pós-poda:
    ao colapsar uma subárvore, a folha herda essa predição).
    Folha = ausência de filhos (left/right is None).
    """

    def __init__(
        self,
        feature=None,
        threshold=None,
        left=None,
        right=None,
        n_samples=0,
        gamma=None
    ):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.n_samples = n_samples
        self.gamma = gamma # Valor da previsão (em log(odds))

    @property
    def is_leaf(self):
        return self.left is None and self.right is None

def build_tree(
    df,
    r, # Resíduos
    p, # Probailidade anterior
    depth=0,
    max_depth=8,
    min_samples_split=2,
):
    """
    Constrói recursivamente uma árvore CART.

    Critérios de parada (folha):
      - nó puro (uma única classe)
      - menos de min_samples_split amostras
      - profundidade >= max_depth (se informado)
      - nenhum split válido restante

    Todo nó (interno ou folha) guarda `prediction` = classe majoritária
    e `n_samples`, necessários para pós-poda (REP / CCP).
    """
    n = len(df)

    gamma = r.sum() / (p * (1-p)).sum()

    too_small = n < min_samples_split
    too_deep = max_depth is not None and depth >= max_depth

    if too_small or too_deep:
        return Node(gamma=gamma, n_samples=n)

    feature, threshold, _ = find_best_split(df, r)
    if feature is None:
        return Node(gamma=gamma, n_samples=n)

    left_mask = df[feature] <= threshold
    right_mask = ~left_mask

    left_df = df[left_mask]
    right_df = df[right_mask]

    # Split degenerado (não deve ocorrer se find_cart_threshold estiver ok)
    if len(left_df) == 0 or len(right_df) == 0:
        return Node(gamma=gamma, n_samples=n)

    left = build_tree(
        left_df,
        p=p[left_mask],
        r=r[left_mask],
        depth=depth + 1,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
    )
    right = build_tree(
        right_df,
        p=p[right_mask],
        r=r[right_mask],
        depth=depth + 1,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
    )

    return Node(
        feature=feature,
        threshold=threshold,
        left=left,
        right=right,
        gamma=None,
        n_samples=n,
    )


def predict_tree_row(node, X):
    if node.is_leaf:
        return node.gamma
    if X[node.feature] <= node.threshold:
        return predict_tree_row(node.left, X)
    return predict_tree_row(node.right, X)

def predict_tree(node, X):
    return np.array([predict_tree_row(node, x) for _, x in X.iterrows()])

# 04. Execução de estratégia de Boosting

In [7]:
n_trees = 10
learning_rate = 0.1
max_depth = 2

y = df_y.copy()
X = df_X.copy()

# 1. Inicializar Log-Odds F0 com constante f0 = log(odds) do dataset
p_mean = np.mean(y)
f_0 = np.log(p_mean / (1 - p_mean))

# Vetor de predição F(x) no espaço Log-Odds
F = np.full(shape=len(y), fill_value=f_0, dtype=float)

trees = []

for m in range(n_trees):
    print(m)
    # a. Calcular probabilidades atuais p_i = sigmoid(F_i)
    p = logistic_function(F)
    
    # b. Calcular resíduos pseudo-gradientes: r_im = y_i - p_i
    r = y - p
    
    # c. Ajustar nova árvore de regressão aos resíduos
    tree = build_tree(X, r, p, max_depth=max_depth)
    
    # d. Obter predição da nova árvore no espaço Log-Odds
    gamma_pred = predict_tree(tree, X)
    
    # e. Atualizar F(x) = F(x) + learning_rate * gamma
    F += learning_rate * gamma_pred
    
    trees.append(tree)
    
    if m % 10 == 0:
        # Calcular Log-Loss para monitoramento
        loss = -np.mean(y * np.log(p + 1e-15) + (1 - y) * np.log(1 - p + 1e-15))
        print(f"Árvore {m:2d} | Log-Loss: {loss:.5f}")

# Teste de Predição de Probabilidades Finais
p_final = logistic_function(F)
y_pred = (p_final >= 0.5).astype(int)
print(f"\nAcurácia no Treino: {np.mean(y_pred == y) * 100:.2f}%")

0
Árvore  0 | Log-Loss: 0.66709
1
2
3
4
5
6
7
8
9

Acurácia no Treino: 78.54%


# Anotações Extras: Gradient Boosting

## 1. O Conceito "Off-the-Shelf" (Pronto para Uso)
* **Origem da propriedade:** A capacidade de exigir pouco pré-processamento **vem da estrutura das Árvores de Decisão (CART)**, que servem como algoritmo base (*weak learner*).
* **Invariância a escalas:** Árvores realizam cortes ordinais do tipo $X_k \le t$. Por isso, são **invariantes a transformações monotônicas** (como $\log(x)$, min-max ou padronização z-score). Modelos de ensemble baseados em árvores (Bagging, Random Forest e Boosting) herdam essa vantagem.
* **Robustez:** Lida nativamente com *outliers* nas variáveis explicativas e é imune a problemas de multicolinearidade.

---

## 2. Gradient Boosting como Framework Genérico
* **Definição:** O Gradient Boosting não é apenas um modelo único, mas uma **metodologia de otimização genérica** baseada em *Gradient Descent* no espaço de funções.
* **Independência de Algoritmo Base:** Embora ~99% das aplicações práticas utilizem Árvores de Decisão (pela capacidade de capturar interações não lineares), o framework pode utilizar outros modelos como aprendedores fracos (*weak learners*), como Regressão Linear ou Splines.
* **Funções de Perda Customizáveis:** A mesma estrutura de Boosting pode otimizar diferentes funções objetivo apenas alterando o cálculo do gradiente:
  * **Regressão:** MSE ($L_2$), MAE ($L_1$), Huber Loss.
  * **Classificação:** Log-Loss (LogitBoost), AdaBoost Exponential Loss.
  * **Outros:** Poisson (dados de contagem), Cox (análise de sobrevivência).

---

## 3. Regularização via Shrinkage (Learning Rate)
* A inclusão da **Taxa de Aprendizado ($\mu \le 0.1$)** atua como um fator de encolhimento (*shrinkage*).
* Em vez de atualizar integralmente o modelo a cada iteração com a nova árvore ($F_m = F_{m-1} + h_m$), aceita-se apenas uma fração do seu aprendizado ($F_m = F_{m-1} + \mu \cdot h_m$).
* Isso força o algoritmo a construir um número maior de árvores "complementares" e rasas, **prevenindo o overfitting** de forma muito mais eficiente do que atualizar com passo unitário ($\mu = 1.0$).

---

## 4. XGBoost (Extreme Gradient Boosting)
* **Não é apenas para datasets grandes:** O XGBoost revolucionou o algoritmo tradicional por implementar **regularização matemática rigorosa** e **otimização de segunda ordem**.
* **Aproximação por Série de Taylor (2ª Ordem):** Enquanto o Gradient Boosting tradicional usa apenas a 1ª derivada (gradiente $g_i$), o XGBoost utiliza a 1ª e a **2ª derivada (Hessiana $h_i$)**, permitindo encontrar o valor ótimo de predição dos nós de forma mais rápida e precisa.
* **Regularização Embutida ($\gamma$ e $\lambda$):** Aplica penalização estilo $L_1$ (Lasso) e $L_2$ (Ridge) diretamente na função de perda para controlar a complexidade das folhas e evitar *overfitting*.
* **Tratamento de Dados Ausentes (*Sparsity-aware*):** Aprende automaticamente a "direção padrão" (*default path*) na árvore para observações com valores nulos/missing.

---

## 5. Comparativo Rápido de Ensembles

| Modelo | Tipo de Aprendedor | Como Reduz o Erro? | Foco Principal |
| :--- | :--- | :--- | :--- |
| **Árvore Única** | Único (Profundo) | N/A | **Nenhum** (Propenso a alto viés se rasa, ou alta variância se profunda). |
| **Random Forest / Bagging** | Paralelo (Profundo) | Média/Votação entre árvores independentes em amostras bootstrap. | **Reduz a Variância** (combate overfitting). |
| **Gradient Boosting** | Sequencial (Raso) | Ajuste sequencial de árvores fracas nos resíduos (gradientes) das anteriores. | **Reduz o Viés** (e a variância via *shrinkage*). |